In [1]:
import pandas as pd
import sqlite3

In [12]:
podatci_zone = [
    {"naziv_zone": "Centar", "tip_korisnika": "Fizičke osobe", "cijena_po_m3": 1.25},
    {"naziv_zone": "Vir", "tip_korisnika": "Fizičke osobe", "cijena_po_m3": 1.25},
    {"naziv_zone": "Rakitno", "tip_korisnika": "Fizičke osobe", "cijena_po_m3": 1.25},
    {"naziv_zone": "Batin", "tip_korisnika": "Fizičke osobe", "cijena_po_m3": 1.25},
    {"naziv_zone": "Poslovna zona", "tip_korisnika": "Pravne osobe", "cijena_po_m3": 2.00},
    {"naziv_zone": "Vinjani", "tip_korisnika": "Fizičke osobe", "cijena_po_m3": 1.25}
]

df_zone = pd.DataFrame(podatci_zone)
conn = sqlite3.connect('vodovod_posusje.db')

df_zone.to_sql('zone', conn, if_exists='replace', index=True)

display(df_zone)
conn.close()
print('Podatci su uspješno spremljeni u SQLite bazu!')


,naziv_zone,tip_korisnika,cijena_po_m3
0,Centar,Fizičke osobe,1.25
1,Vir,Fizičke osobe,1.25
2,Rakitno,Fizičke osobe,1.25
3,Batin,Fizičke osobe,1.25
4,Poslovna zona,Pravne osobe,2.00
5,Vinjani,Fizičke osobe,1.25


Podatci su uspješno spremljeni u SQLite bazu!


In [ ]:
import random

In [24]:
zone = ["Centar", "Vir", "Rakitno", "Batin", "Poslovna zona", "Vinjani"]
mjeseci = ["Siječanj", "Veljača", "Ožujak", "Travanj", "Svibanj", "Lipanj", 
           "Srpanj", "Kolovoz", "Rujan", "Listopad", "Studeni", "Prosinac"]

podatci_potrosnje = []

for zona in zone:
    for mjesec in mjeseci:
        osnovna_potrosnja = 15.0

        if mjesec == "Srpanj" or mjesec == "Kolovoz":
            mjesecna_potrosnja = osnovna_potrosnja + (osnovna_potrosnja*0.5)
        else:
            mjesecna_potrosnja = osnovna_potrosnja
            
        podatci_potrosnje.append({
            "naziv_zone" : zona,
            "mjesec": mjesec,
            "prosjek_po_kucanstvu_m3": mjesecna_potrosnja
        })
df_potrosnja =pd.DataFrame(podatci_potrosnje)
df_potrosnja.to_sql('potrosnja', conn, if_exists='replace', index=False)
display(df_potrosnja)


        

,naziv_zone,mjesec,prosjek_po_kucanstvu_m3
0,Centar,Siječanj,15.0
1,Centar,Veljača,15.0
2,Centar,Ožujak,15.0
3,Centar,Travanj,15.0
4,Centar,Svibanj,15.0
...,...,...,...
67,Vinjani,Kolovoz,22.5
68,Vinjani,Rujan,15.0
69,Vinjani,Listopad,15.0
70,Vinjani,Studeni,15.0


In [26]:
upit = """
    SELECT z.naziv_zone,z.tip_korisnika, z.cijena_po_m3, p.mjesec, p.prosjek_po_kucanstvu_m3
    FROM zone z
    JOIN potrosnja p ON z.naziv_zone=p.naziv_zone """

conn = sqlite3.connect('vodovod_posusje.db')
df_rezultat = pd.read_sql(upit, conn)

conn.close()

# Financijski izračun
df_rezultat['iznos_voda'] = df_rezultat['prosjek_po_kucanstvu_m3'] * df_rezultat['cijena_po_m3']
df_rezultat['odrzavanje_vodomjera'] = 3.00
df_rezultat['vodne_naknade'] = df_rezultat['prosjek_po_kucanstvu_m3'] * (0.01 + 0.04)
df_rezultat['ukupno_za_platiti'] = df_rezultat['iznos_voda'] + df_rezultat['odrzavanje_vodomjera'] + df_rezultat['vodne_naknade']

display(df_rezultat.head(8))

,naziv_zone,tip_korisnika,cijena_po_m3,mjesec,prosjek_po_kucanstvu_m3,iznos_voda,odrzavanje_vodomjera,vodne_naknade,ukupno_za_platiti
0,Centar,Fizičke osobe,1.25,Kolovoz,22.5,28.125,3.0,1.125,32.25
1,Centar,Fizičke osobe,1.25,Lipanj,15.0,18.750,3.0,0.750,22.50
2,Centar,Fizičke osobe,1.25,Listopad,15.0,18.750,3.0,0.750,22.50
3,Centar,Fizičke osobe,1.25,Ožujak,15.0,18.750,3.0,0.750,22.50
4,Centar,Fizičke osobe,1.25,Prosinac,15.0,18.750,3.0,0.750,22.50
5,Centar,Fizičke osobe,1.25,Rujan,15.0,18.750,3.0,0.750,22.50
6,Centar,Fizičke osobe,1.25,Siječanj,15.0,18.750,3.0,0.750,22.50
7,Centar,Fizičke osobe,1.25,Srpanj,22.5,28.125,3.0,1.125,32.25
